<a href="https://colab.research.google.com/github/ghada-dahdoh/Applied-natural-language-processing/blob/main/Lab_3B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "CAMeL-Lab/bert-base-arabic-camelbert-mix"
)

words = [
    "التقى",
    "عبدالله",
    "بمدير",
    "جامعة",
    "الملك",
    "سلمان",
    "في",
    "الدمام"
]

labels = [
    "O",
    "B-PER",
    "O",
    "B-ORG",
    "I-ORG",
    "I-ORG",
    "O",
    "B-LOC"
]

tokens = tokenizer(
    words,
    is_split_into_words=True
)

print("=== TOKENIZATION ===")
print("Words:", words)

print("\nTokens:")
print(
    tokenizer.convert_ids_to_tokens(
        tokens["input_ids"]
    )
)

print("\nWord IDs:")
print(tokens.word_ids())

=== TOKENIZATION ===
Words: ['التقى', 'عبدالله', 'بمدير', 'جامعة', 'الملك', 'سلمان', 'في', 'الدمام']

Tokens:
['[CLS]', 'التقى', 'عبدالله', 'بمد', '##ير', 'جامعة', 'الملك', 'سلمان', 'في', 'الدمام', '[SEP]']

Word IDs:
[None, 0, 1, 2, 2, 3, 4, 5, 6, 7, None]


In [ ]:
token_list = tokenizer.convert_ids_to_tokens(
    tokens["input_ids"]
)

word_ids = tokens.word_ids()

print("\n=== SUBWORD ALIGNMENT ===")

for i, (token, word_id) in enumerate(
    zip(token_list, word_ids)
):
    if word_id is None:
        original_word = "SPECIAL TOKEN"
    else:
        original_word = words[word_id]

    print(
        f"Token {i:02d} | "
        f"{token:<15} | "
        f"Word ID: {str(word_id):<4} | "
        f"Word: {original_word}"
    )


=== SUBWORD ALIGNMENT ===
Token 00 | [CLS]           | Word ID: None | Word: SPECIAL TOKEN
Token 01 | التقى           | Word ID: 0    | Word: التقى
Token 02 | عبدالله         | Word ID: 1    | Word: عبدالله
Token 03 | بمد             | Word ID: 2    | Word: بمدير
Token 04 | ##ير            | Word ID: 2    | Word: بمدير
Token 05 | جامعة           | Word ID: 3    | Word: جامعة
Token 06 | الملك           | Word ID: 4    | Word: الملك
Token 07 | سلمان           | Word ID: 5    | Word: سلمان
Token 08 | في              | Word ID: 6    | Word: في
Token 09 | الدمام          | Word ID: 7    | Word: الدمام
Token 10 | [SEP]           | Word ID: None | Word: SPECIAL TOKEN


In [ ]:
print("\n=== SUBWORD TYPE ===")

previous_word_id = None

for token, word_id in zip(
    token_list,
    word_ids
):
    if word_id is None:
        continue

    if word_id != previous_word_id:
        subword_type = "FIRST SUBWORD"
    else:
        subword_type = "CONTINUATION"

    print(
        f"{token:<15} → "
        f"Word {word_id} → "
        f"{subword_type}"
    )

    previous_word_id = word_id


=== SUBWORD TYPE ===
التقى           → Word 0 → FIRST SUBWORD
عبدالله         → Word 1 → FIRST SUBWORD
بمد             → Word 2 → FIRST SUBWORD
##ير            → Word 2 → CONTINUATION
جامعة           → Word 3 → FIRST SUBWORD
الملك           → Word 4 → FIRST SUBWORD
سلمان           → Word 5 → FIRST SUBWORD
في              → Word 6 → FIRST SUBWORD
الدمام          → Word 7 → FIRST SUBWORD


In [ ]:
clitic_words = [
    "والجامعة",
    "بالرياض",
    "ومديرها",
    "للطلاب",
    "فالمؤتمر",
    "وبالمدرسة"
]

clitic_tokens = tokenizer(
    clitic_words,
    is_split_into_words=True
)

print("\n=== ARABIC CLITIC ANALYSIS ===")

for token, word_id in zip(
    tokenizer.convert_ids_to_tokens(
        clitic_tokens["input_ids"]
    ),
    clitic_tokens.word_ids()
):
    if word_id is None:
        print(f"{token:<15} → SPECIAL TOKEN")
    else:
        print(
            f"{token:<15} → "
            f"{clitic_words[word_id]}"
        )


=== ARABIC CLITIC ANALYSIS ===
[CLS]           → SPECIAL TOKEN
والج            → والجامعة
##امعة          → والجامعة
بالرياض         → بالرياض
ومدير           → ومديرها
##ها            → ومديرها
للطلاب          → للطلاب
فالم            → فالمؤتمر
##ؤ             → فالمؤتمر
##تمر           → فالمؤتمر
وبالم           → وبالمدرسة
##درسة          → وبالمدرسة
[SEP]           → SPECIAL TOKEN


In [ ]:
print("\n=== NER LABEL ALIGNMENT ===")

for token, word_id in zip(
    token_list,
    word_ids
):
    if word_id is None:
        print(
            f"{token:<15} | "
            f"SPECIAL TOKEN | SPECIAL"
        )
    else:
        print(
            f"{token:<15} | "
            f"{words[word_id]:<12} | "
            f"{labels[word_id]}"
        )


=== NER LABEL ALIGNMENT ===
[CLS]           | SPECIAL TOKEN | SPECIAL
التقى           | التقى        | O
عبدالله         | عبدالله      | B-PER
بمد             | بمدير        | O
##ير            | بمدير        | O
جامعة           | جامعة        | B-ORG
الملك           | الملك        | I-ORG
سلمان           | سلمان        | I-ORG
في              | في           | O
الدمام          | الدمام       | B-LOC
[SEP]           | SPECIAL TOKEN | SPECIAL


In [ ]:
def check_alignment(words, labels, tokenizer):

    encoded = tokenizer(
        words,
        is_split_into_words=True
    )

    word_ids = encoded.word_ids()

    detected_ids = {
        wid for wid in word_ids
        if wid is not None
    }

    expected_ids = set(
        range(len(words))
    )

    missing_ids = expected_ids - detected_ids

    print("\n=== ALIGNMENT QUALITY CHECK ===")

    print(
        "Words:",
        len(words)
    )

    print(
        "Tokens:",
        len(encoded["input_ids"])
    )

    if len(words) == len(labels):
        print("✓ Words and labels match.")
    else:
        print("✗ Words and labels do not match.")

    if not missing_ids:
        print("✓ All words successfully aligned.")
        return True
    else:
        print(
            "✗ Missing Word IDs:",
            missing_ids
        )
        return False


alignment_result = check_alignment(
    words,
    labels,
    tokenizer
)


=== ALIGNMENT QUALITY CHECK ===
Words: 8
Tokens: 11
✓ Words and labels match.
✓ All words successfully aligned.


In [ ]:
tests = [
    (
        ["وصل", "سلمان", "إلى", "مكة"],
        ["O", "B-PER", "O", "B-LOC"]
    ),

    (
        ["زارت", "نورة", "جامعة", "الملك"],
        ["O", "B-PER", "B-ORG", "I-ORG"]
    ),

    (
        ["يقع", "المطار", "في", "جدة"],
        ["O", "O", "O", "B-LOC"]
    ),

    (
        ["حضر", "خالد", "المؤتمر"],
        ["O", "B-PER", "O"]
    ),

    (
        ["شركة", "أرامكو", "في", "الظهران"],
        ["O", "B-ORG", "O", "B-LOC"]
    ),

    (
        ["تحدثت", "سارة", "مع", "المدير"],
        ["O", "B-PER", "O", "O"]
    ),

    (
        ["جامعة", "الملك", "عبدالعزيز"],
        ["B-ORG", "I-ORG", "I-ORG"]
    ),

    (
        ["سافر", "أحمد", "إلى", "المدينة"],
        ["O", "B-PER", "O", "B-LOC"]
    )
]

passed = 0

print("\n=== ALIGNMENT TESTS ===")

for i, (test_words, test_labels) in enumerate(
    tests,
    start=1
):

    result = check_alignment(
        test_words,
        test_labels,
        tokenizer
    )

    if result:
        passed += 1

    print(
        f"Test {i}: "
        f"{'PASS ✓' if result else 'FAIL ✗'}"
    )

print("\n----------------------------")

print(
    f"Tests Passed: "
    f"{passed}/{len(tests)}"
)

print(
    f"Success Rate: "
    f"{passed / len(tests) * 100:.1f}%"
)


=== ALIGNMENT TESTS ===

=== ALIGNMENT QUALITY CHECK ===
Words: 4
Tokens: 6
✓ Words and labels match.
✓ All words successfully aligned.
Test 1: PASS ✓

=== ALIGNMENT QUALITY CHECK ===
Words: 4
Tokens: 8
✓ Words and labels match.
✓ All words successfully aligned.
Test 2: PASS ✓

=== ALIGNMENT QUALITY CHECK ===
Words: 4
Tokens: 6
✓ Words and labels match.
✓ All words successfully aligned.
Test 3: PASS ✓

=== ALIGNMENT QUALITY CHECK ===
Words: 3
Tokens: 5
✓ Words and labels match.
✓ All words successfully aligned.
Test 4: PASS ✓

=== ALIGNMENT QUALITY CHECK ===
Words: 4
Tokens: 9
✓ Words and labels match.
✓ All words successfully aligned.
Test 5: PASS ✓

=== ALIGNMENT QUALITY CHECK ===
Words: 4
Tokens: 6
✓ Words and labels match.
✓ All words successfully aligned.
Test 6: PASS ✓

=== ALIGNMENT QUALITY CHECK ===
Words: 3
Tokens: 5
✓ Words and labels match.
✓ All words successfully aligned.
Test 7: PASS ✓

=== ALIGNMENT QUALITY CHECK ===
Words: 4
Tokens: 6
✓ Words and labels match.
✓ All wo

In [ ]:
y_true = [
    "O",
    "B-PER",
    "O",
    "B-ORG",
    "I-ORG",
    "I-ORG",
    "O",
    "B-LOC"
]

y_pred = [
    "O",
    "B-PER",
    "O",
    "B-ORG",
    "I-ORG",
    "I-ORG",
    "O",
    "B-LOC"
]

correct = sum(
    true == pred
    for true, pred in zip(y_true, y_pred)
)

total = len(y_true)

accuracy = correct / total

print("\n=== NER EVALUATION ===")

print(
    f"Correct Predictions: {correct}/{total}"
)

print(
    f"NER Accuracy: {accuracy * 100:.1f}%"
)


=== NER EVALUATION ===
Correct Predictions: 8/8
NER Accuracy: 100.0%


In [ ]:
print("\n" + "=" * 60)
print("        ARABIC NER ALIGNMENT REPORT")
print("=" * 60)

print("Tokenizer       : CAMeL-BERT Arabic")
print(f"Input Words     : {len(words)}")
print(f"Total Tokens    : {len(tokens['input_ids'])}")
print(f"Alignment Tests : {passed}/{len(tests)}")
print(
    f"Success Rate    : "
    f"{passed / len(tests) * 100:.1f}%"
)
print(
    f"NER Accuracy    : "
    f"{accuracy * 100:.1f}%"
)

print("=" * 60)

if alignment_result and passed == len(tests):
    print(
        "✓ Alignment Pipeline "
        "Completed Successfully"
    )
else:
    print(
        "⚠ Alignment Pipeline "
        "Requires Review"
    )

print("=" * 60)


        ARABIC NER ALIGNMENT REPORT
Tokenizer       : CAMeL-BERT Arabic
Input Words     : 8
Total Tokens    : 11
Alignment Tests : 8/8
Success Rate    : 100.0%
NER Accuracy    : 100.0%
✓ Alignment Pipeline Completed Successfully
